<a href="https://colab.research.google.com/github/misrori/ai/blob/2025/leiratozo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai==0.28
!pip install openai pydub


In [ ]:
import os
import time
import json
from datetime import timedelta
import openai
from pydub import AudioSegment
import math

# OpenAI API kulcs beállítása
openai.api_key = "sk-proj-"


# Maximum fájlméret byte-ban (25MB - kicsit kisebbre állítva biztonság kedvéért)
MAX_FILE_SIZE = 25 * 1024 * 1024

def format_time(seconds):
    """Másodperceket SRT időformátumba konvertál (HH:MM:SS,mmm)"""
    td = timedelta(seconds=seconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int(td.microseconds / 1000)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

def split_audio(audio_file_path, chunk_length_ms=600000):
    """Hangfájl felosztása kisebb darabokra

    Args:
        audio_file_path: A hangfájl útvonala
        chunk_length_ms: Egy részlet hossza millimásodpercben (alapértelmezett: 10 perc)

    Returns:
        temp_files: Ideiglenes fájlok listája
        total_duration_ms: A teljes hangfájl hossza ezredmásodpercben
    """
    print(f"Hangfájl darabolása: {audio_file_path}")

    # Hangfájl betöltése
    audio = AudioSegment.from_file(audio_file_path)
    total_duration_ms = len(audio)

    # Hány darabra kell osztani
    file_size = os.path.getsize(audio_file_path)
    num_chunks = math.ceil(file_size / MAX_FILE_SIZE)
    chunk_length_ms = min(chunk_length_ms, math.ceil(total_duration_ms / num_chunks))

    # Darabolás és ideiglenes fájlok mentése
    temp_files = []
    for i, chunk_start in enumerate(range(0, len(audio), chunk_length_ms)):
        chunk_end = min(chunk_start + chunk_length_ms, len(audio))
        chunk = audio[chunk_start:chunk_end]

        # Ideiglenes fájl készítése
        chunk_file = f"temp_chunk_{i}.mp3"
        chunk.export(chunk_file, format="mp3")
        temp_files.append((chunk_file, chunk_start / 1000))  # Kezdő idő másodpercben

        print(f"Részlet {i+1} mentve: {chunk_file}")

    return temp_files, total_duration_ms / 1000

def transcribe_audio_chunk(audio_chunk_path):
    """Egy hangfájl részlet átírása"""
    try:
        with open(audio_chunk_path, "rb") as audio_file:
            transcription = openai.Audio.transcribe(
                "whisper-1",
                audio_file,
                response_format="verbose_json"
            )
        return transcription
    except Exception as e:
        print(f"Hiba történt a részlet átírása során: {str(e)}")
        return None

def create_srt_from_audio(audio_file_path, output_srt_path):
    """Hangfájlból SRT feliratot készít, szükség esetén darabolva"""
    print(f"Hangfájl feldolgozása: {audio_file_path}")

    # A fájlméret ellenőrzése
    file_size = os.path.getsize(audio_file_path)

    if file_size <= MAX_FILE_SIZE:
        # Ha a fájl mérete megfelelő, egyszerűen átírjuk
        try:
            with open(audio_file_path, "rb") as audio_file:
                transcription = openai.Audio.transcribe(
                    "whisper-1",
                    audio_file,
                    response_format="verbose_json"
                )

            # SRT fájl készítése
            create_srt_from_transcription(transcription, output_srt_path)
        except Exception as e:
            print(f"Hiba történt az átírás során: {str(e)}")
            return
    else:
        # Ha túl nagy a fájl, feldaraboljuk és részenként írjuk át
        temp_chunks, total_duration = split_audio(audio_file_path)

        # SRT fájl előkészítése
        with open(output_srt_path, "w", encoding="utf-8") as srt_file:
            subtitle_index = 1

            for chunk_file, start_offset in temp_chunks:
                transcription = transcribe_audio_chunk(chunk_file)

                if transcription and hasattr(transcription, "segments"):
                    # Részletek hozzáadása
                    for segment in transcription.segments:
                        start_time = segment.start + start_offset
                        end_time = segment.end + start_offset
                        text = segment.text.strip()

                        # SRT bejegyzés írása
                        srt_file.write(f"{subtitle_index}\n")
                        srt_file.write(f"{format_time(start_time)} --> {format_time(end_time)}\n")
                        srt_file.write(f"{text}\n\n")

                        subtitle_index += 1

                # Ideiglenes fájl törlése
                os.remove(chunk_file)

        print(f"SRT felirat sikeresen elkészült: {output_srt_path}")

def create_srt_from_transcription(transcription, output_path):
    """Átírásból SRT fájlt készít"""
    with open(output_path, "w", encoding="utf-8") as srt_file:
        if hasattr(transcription, "segments"):
            for i, segment in enumerate(transcription.segments):
                start_time = segment.start
                end_time = segment.end
                text = segment.text.strip()

                # SRT bejegyzés írása
                srt_file.write(f"{i+1}\n")
                srt_file.write(f"{format_time(start_time)} --> {format_time(end_time)}\n")
                srt_file.write(f"{text}\n\n")
        else:
            # Ha nincs időzítés, egyszerű SRT készítése
            srt_file.write("1\n")
            srt_file.write("00:00:00,000 --> 00:05:00,000\n")
            srt_file.write(transcription.text + "\n\n")

    print(f"SRT felirat sikeresen elkészült: {output_path}")

# Példa használat
if __name__ == "__main__":
    audio_file_path = "viktor.mp3"  # Ide írd a hangfájl elérési útját
    output_srt_path = "felirat.srt"   # Az elkészített SRT fájl neve

    create_srt_from_audio(audio_file_path, output_srt_path)

Hangfájl feldolgozása: viktor.mp3
Hangfájl darabolása: viktor.mp3
Részlet 1 mentve: temp_chunk_0.mp3
Részlet 2 mentve: temp_chunk_1.mp3
Részlet 3 mentve: temp_chunk_2.mp3
SRT felirat sikeresen elkészült: felirat.srt
